# 06 · Awards: You vs the Machine

Predict the season's awards from a point in the middle of the season (`AFTER_GW`). Each correct award is worth **5 points**.

| Award | How the winner is decided here |
|---|---|
| Player of the Week | most FPL points in a gameweek |
| Player of the Season | most FPL points over the season |
| Golden Boot | most goals |
| Playmaker | most assists (FPL counts assists slightly differently from the official award) |
| Golden Glove | goalkeeper with the most clean sheets |

The real Player of the Season is chosen by a vote, and there have only been a handful of winners, which is far too few to
train a model on. So here it means "best FPL season", which can be measured.

**How the machine projects the rest of the season** (`pl.award_projections`):
- *so far* + *rate per team match so far* × *remaining fixtures*
- goals and assists are scaled by how many goals the team is expected to score in its remaining fixtures (Poisson model)
- clean sheets add up P(clean sheet) = e^(−expected goals conceded) over the remaining fixtures, times how often the keeper plays
- team form and Elo are frozen at `AFTER_GW`, so no future results are used

In [2]:
#Import libraries and load the helper file (src/pl.py)
import sys
from pathlib import Path

#This notebook runs from the notebooks folder, so the project folder is one level up ("..")
project_folder = str(Path("..").resolve())
sys.path.insert(0, project_folder)
from src import pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Let printed tables be wider before they wrap
pd.set_option("display.width", 140)

In [3]:
#Settings
SEASON = "2025-26"     #the season to play
AFTER_GW = 19          #make the predictions after this gameweek (19 = halfway)
PLAYER = "You"         #my name in the results

### Load the data

In [4]:
#Load the match data: match table, one row per team per match, and Elo ratings over time
mdf, team_games, elo_hist = pl.build_match_table()

#Expected goals for every match (each season is predicted using only earlier seasons)
lam = pl.walk_forward_lambdas(mdf)

#Load the FPL player data
fpl = pl.load_fpl()

#Add match info (Elo, expected goals) to every player row
fpl_with_context = pl.add_fixture_context(fpl, lam)

#Make one row per player per gameweek
pgw = pl.build_player_gameweeks(fpl_with_context)

#Make the fixture list, then add the No_Crowd column from the match table
fixtures = pl.fpl_fixtures(fpl)
crowd_info = mdf[["Season", "Home_Team", "Away_Team", "No_Crowd"]]
fixtures = fixtures.merge(crowd_info, on=["Season", "Home_Team", "Away_Team"])

### The machine's projections

In [5]:
#Train the goal models on the seasons before SEASON only
earlier_seasons = mdf[mdf["Season"] < SEASON]
goal_models = pl.fit_goal_models(earlier_seasons)

#Project every player's end-of-season totals using only gameweeks 1 to AFTER_GW
proj = pl.award_projections(pgw, fixtures, team_games, elo_hist, goal_models, SEASON, AFTER_GW)

In [6]:
#Player of the Season: top 5 by projected points
top5 = proj.sort_values("Proj_Points", ascending=False)
top5 = top5.head(5)
top5 = top5[["Name", "Team", "Points", "Proj_Points"]]
top5 = top5.round(1)
print(f"--- Player of the Season: top 5 projected after GW {AFTER_GW}")
print(top5.to_string(index=False))

#Golden Boot: top 5 by projected goals
top5 = proj.sort_values("Proj_Goals", ascending=False)
top5 = top5.head(5)
top5 = top5[["Name", "Team", "Goals", "Proj_Goals"]]
top5 = top5.round(1)
print(f"--- Golden Boot: top 5 projected after GW {AFTER_GW}")
print(top5.to_string(index=False))

#Playmaker: top 5 by projected assists
top5 = proj.sort_values("Proj_Assists", ascending=False)
top5 = top5.head(5)
top5 = top5[["Name", "Team", "Assists", "Proj_Assists"]]
top5 = top5.round(1)
print(f"--- Playmaker: top 5 projected after GW {AFTER_GW}")
print(top5.to_string(index=False))

#Golden Glove: top 5 by projected clean sheets (goalkeepers only)
top5 = proj.sort_values("Proj_Clean_Sheets", ascending=False)
top5 = top5.head(5)
top5 = top5[["Name", "Team", "Clean_Sheets", "Proj_Clean_Sheets"]]
top5 = top5.round(1)
print(f"--- Golden Glove: top 5 projected after GW {AFTER_GW}")
print(top5.to_string(index=False))

--- Player of the Season: top 5 projected after GW 19
                        Name        Team  Points  Proj_Points
              Erling Haaland    Man City     155        310.0
             Antoine Semenyo Bournemouth     113        226.0
                 Declan Rice     Arsenal     102        204.0
      Bruno Borges Fernandes  Man United      99        198.0
Gabriel dos Santos Magalhães     Arsenal      97        194.0
--- Golden Boot: top 5 projected after GW 19
                            Name        Team  Goals  Proj_Goals
                  Erling Haaland    Man City     19        45.3
Igor Thiago Nascimento Rodrigues   Brentford     11        20.8
                 Antoine Semenyo Bournemouth      9        18.6
                    Hugo Ekitiké   Liverpool      8        17.9
                      Phil Foden    Man City      7        16.7
--- Playmaker: top 5 projected after GW 19
                  Name       Team  Assists  Proj_Assists
          Rayan Cherki   Man City        8   

### Your award picks

In [7]:
#My picks (accents and capital letters don't matter)
my_awards = {
    "Player of the Season": "Mohamed Salah",
    "Golden Boot": "Erling Haaland",
    "Playmaker": "Bukayo Saka",
    "Golden Glove": "David Raya Martín",
}

### Reveal

In [8]:
#Get every player who played this season (one row per player)
season_players = pgw[pgw["Season"] == SEASON]
season_players = season_players.drop_duplicates("Player_ID")

#Turn the names I typed into the official full names
my_typed_names = list(my_awards.values())
my_ids = pl.resolve_players(season_players, my_typed_names)
players_by_id = season_players.set_index("Player_ID")

names = {}
names["Player of the Season"] = players_by_id.loc[my_ids[0], "Name"]
names["Golden Boot"] = players_by_id.loc[my_ids[1], "Name"]
names["Playmaker"] = players_by_id.loc[my_ids[2], "Name"]
names["Golden Glove"] = players_by_id.loc[my_ids[3], "Name"]

#Get the machine's picks (highest projection) and the real winners
machine = pl.machine_award_picks(proj)
winners = pl.season_winners(pgw, SEASON)

#Check each award: 5 points for a correct pick, 0 if wrong
rows = []
for award in pl.AWARDS:
    my_choice = names[award]
    machine_choice = machine[award]
    real_winners = winners[award]

    if my_choice in real_winners:
        my_points = 5
    else:
        my_points = 0

    if machine_choice in real_winners:
        machine_points = 5
    else:
        machine_points = 0

    #If players tie, show them all as "A / B"
    winner_text = " / ".join(sorted(real_winners))

    rows.append({"Award": award, "You": my_choice, "Machine": machine_choice, "Winner": winner_text,
                 "Your pts": my_points, "Machine pts": machine_points})

reveal = pd.DataFrame(rows)

#Total points
my_total = reveal["Your pts"].sum()
machine_total = reveal["Machine pts"].sum()
print(f"{SEASON} awards:  {PLAYER} {my_total} pts  |  Machine {machine_total} pts")
reveal

2025-26 awards:  You 10 pts  |  Machine 15 pts


,Award,You,Machine,Winner,Your pts,Machine pts
0,Player of the Season,Mohamed Salah,Erling Haaland,Erling Haaland,0,5
1,Golden Boot,Erling Haaland,Erling Haaland,Erling Haaland,5,5
2,Playmaker,Bukayo Saka,Rayan Cherki,Bruno Borges Fernandes,0,0
3,Golden Glove,David Raya Martín,David Raya Martín,David Raya Martín,5,5


In [9]:
#Make a table of my picks and points
my_log = pd.DataFrame({"Season": SEASON, "After_GW": AFTER_GW, "Award": reveal["Award"], "Player": PLAYER,
                       "Pick": reveal["You"], "Points": reveal["Your pts"]})

#Make a table of the machine's picks and points
machine_log = pd.DataFrame({"Season": SEASON, "After_GW": AFTER_GW, "Award": reveal["Award"], "Player": "Machine",
                            "Pick": reveal["Machine"], "Points": reveal["Machine pts"]})

#Put both tables together (mine on top, the machine's below)
log = pd.concat([my_log, machine_log])

#Save to game/awards.csv (playing the same game again replaces the old rows instead of adding copies)
history = pl.save_results(log, "awards.csv", keys=["Season", "After_GW", "Award", "Player"])

#Total points for each player over all saved games
total_points = history.groupby("Player")["Points"].sum()
print(total_points.to_dict())

{'Machine': 15, 'You': 10}


## Player of the Week

Pick who you think will score the most FPL points in `POTW_GW`. The machine picks the player with the highest
predicted points from the fantasy model in notebook 05.

In [10]:
#Player of the Week is the gameweek right after AFTER_GW
POTW_GW = AFTER_GW + 1
my_potw = "Erling Haaland"

#Train the player points model on the seasons before SEASON only
model = pl.fit_player_model(pgw, SEASON)

#Predict points for every player in every gameweek of this season
season_rows = pgw[pgw["Season"] == SEASON].copy()
season_rows["Pred"] = model.predict(season_rows[pl.PLAYER_FEATURES])

In [11]:
#Function: for one gameweek, give back the machine's pick, the real winner(s), and that gameweek's table
def potw(gw):
    #All players in this gameweek
    g = season_rows[season_rows["GW"] == gw]

    #Machine's pick = the player with the highest predicted points
    g_sorted = g.sort_values("Pred", ascending=False)
    machine_pick = g_sorted.iloc[0]

    #Real winner(s) = everyone with the highest actual points (ties all count)
    top = g["Points"].max()
    top_players = g[g["Points"] == top]
    week_winners = set(top_players["Name"])

    return machine_pick, week_winners, g

In [12]:
#Run it for POTW_GW
machine_pick, potw_winners, g = potw(POTW_GW)

#Find my player in that gameweek
my_potw_id = pl.resolve_players(g, [my_potw])[0]
g_by_id = g.set_index("Player_ID")
my_pick = g_by_id.loc[my_potw_id]

#5 points if correct, 0 if wrong
if my_pick["Name"] in potw_winners:
    my_potw_points = 5
else:
    my_potw_points = 0

if machine_pick["Name"] in potw_winners:
    machine_potw_points = 5
else:
    machine_potw_points = 0

#Show the result
winner_text = " / ".join(potw_winners)
top_points = g["Points"].max()
print(f"GW {POTW_GW} Player of the Week: {winner_text} ({top_points} pts)")
print(f"  {PLAYER}: {my_pick['Name']} scored {my_pick['Points']} -> {my_potw_points} pts")
print(f"  Machine: {machine_pick['Name']} scored {machine_pick['Points']} -> {machine_potw_points} pts")

GW 20 Player of the Week: Malick Thiaw / Declan Rice / Igor Thiago Nascimento Rodrigues (17 pts)
  You: Erling Haaland scored 2 -> 0 pts
  Machine: Erling Haaland scored 2 -> 0 pts


In [13]:
#Check every gameweek: did the machine's pick win, or at least finish in the top 10?
rows = []
all_gws = sorted(season_rows["GW"].unique())
for gw in all_gws:
    pick, win, g = potw(gw)

    #Rank of the machine's pick = how many players scored more than him, plus 1
    players_above = (g["Points"] > pick["Points"]).sum()
    rank = players_above + 1

    winner_text = " / ".join(win)
    winner_points = g["Points"].max()

    rows.append({"GW": gw, "Machine pick": pick["Name"], "Pick pts": pick["Points"],
                 "Winner": winner_text, "Winner pts": winner_points, "Pick rank": rank})

potw_table = pd.DataFrame(rows)

#Count the wins (rank 1) and top 10 finishes
wins = (potw_table["Pick rank"] == 1).sum()
top10 = (potw_table["Pick rank"] <= 10).sum()
total_gws = len(potw_table)
print(f"Machine picked the Player of the Week in {wins} of {total_gws} gameweeks; top 10 in {top10}")
potw_table.head(10)

Machine picked the Player of the Week in 0 of 38 gameweeks; top 10 in 13


,GW,Machine pick,Pick pts,Winner,Winner pts,Pick rank
0,1,Jarrod Bowen,2,Daniel Ballard,17,127
1,2,Mohamed Salah,5,Jurriën Timber,24,59
2,3,Bryan Mbeumo,10,Marc Guéhi,15,9
3,4,Mohamed Salah,9,Martín Zubimendi Ibáñez,16,13
4,5,Mohamed Salah,5,Anton Stach / Ryan Gravenberch,15,44
5,6,Erling Haaland,16,Omar Alderete,17,2
6,7,Erling Haaland,8,Antoine Semenyo,18,20
7,8,Erling Haaland,13,Reece James,18,6
8,9,Erling Haaland,2,Micky van de Ven,23,104
9,10,Erling Haaland,13,Diego Gómez Amarilla,16,3


## Backtest: how often is the machine right at halfway?

For every playable season, the machine makes its award picks after gameweek 19, using only that season's first
19 gameweeks and models trained on earlier seasons. As a sanity check, a projection made after gameweek 38 must equal
the real season totals.

In [14]:
#Backtest: for every past season, let the machine pick after GW 19 and compare with the real winners
rows = []
for season in pl.PLAYABLE_SEASONS:
    #Train the goal models only on the seasons before this one
    earlier = mdf[mdf["Season"] < season]
    gm = pl.fit_goal_models(earlier)

    #Machine's picks after GW 19
    proj_19 = pl.award_projections(pgw, fixtures, team_games, elo_hist, gm, season, 19)
    picks = pl.machine_award_picks(proj_19)

    #Real winners
    win = pl.season_winners(pgw, season)

    #Mark each pick with a tick (correct) or a cross (wrong)
    row = {"Season": season}
    for a in pl.AWARDS:
        if picks[a] in win[a]:
            row[a] = "✔ " + picks[a]
        else:
            row[a] = "✘ " + picks[a]
    rows.append(row)

    #Sanity check: after GW 38 there is nothing left to predict, so the machine must match the real winners
    proj_38 = pl.award_projections(pgw, fixtures, team_games, elo_hist, gm, season, 38)
    picks_38 = pl.machine_award_picks(proj_38)
    assert picks_38["Golden Boot"] in win["Golden Boot"]
    assert picks_38["Golden Glove"] in win["Golden Glove"]

backtest = pd.DataFrame(rows)

#Count the ticks in each award column
hits = {}
for a in pl.AWARDS:
    hits[a] = backtest[a].str.startswith("✔").sum()
print("Correct at GW 19 (out of", len(backtest), "seasons):", hits)
backtest

Correct at GW 19 (out of 9 seasons): {'Player of the Season': np.int64(6), 'Golden Boot': np.int64(8), 'Playmaker': np.int64(3), 'Golden Glove': np.int64(6)}


,Season,Player of the Season,Golden Boot,Playmaker,Golden Glove
0,2017-18,✔ Mohamed Salah,✔ Mohamed Salah,✘ Leroy Sané,✘ Thibaut Courtois
1,2018-19,✔ Mohamed Salah,✔ Mohamed Salah,✘ Leroy Sané,✔ Alisson Ramses Becker
2,2019-20,✘ Jamie Vardy,✔ Jamie Vardy,✔ Kevin De Bruyne,✔ Ederson Santana de Moraes
3,2020-21,✘ Harry Kane,✘ Mohamed Salah,✘ Kevin De Bruyne,✔ Ederson Santana de Moraes
4,2021-22,✔ Mohamed Salah,✔ Mohamed Salah,✘ Mohamed Salah,✔ Ederson Santana de Moraes
5,2022-23,✔ Erling Haaland,✔ Erling Haaland,✔ Kevin De Bruyne,✘ Nick Pope
6,2023-24,✘ Mohamed Salah,✔ Erling Haaland,✘ Julián Álvarez,✘ Alisson Ramses Becker
7,2024-25,✔ Mohamed Salah,✔ Mohamed Salah,✔ Mohamed Salah,✔ David Raya Martin
8,2025-26,✔ Erling Haaland,✔ Erling Haaland,✘ Rayan Cherki,✔ David Raya Martín
